In [1]:
#IMPORTING stuff including my online repo with useful tools for processing chess files (PGN)

import sys
import os, sys
!pip install chess mlflow
if os.path.exists("/kaggle/working/ludus_scacchi"):
    print("Repository already exists, skipping clone.")
    sys.path.append("/kaggle/working/ludus_scacchi")
else:
    !git clone https://github.com/paolomostardi/ludus_scacchi.git /kaggle/working/ludus_scacchi
    sys.path.append("/kaggle/working/ludus_scacchi")
from pathlib import Path
from Backend.data_pipeline import from_PGN_generate_bitboards
from Backend.data_pipeline import gm_pipeline as pipe
import pandas as pd
import numpy as np
import keras
import mlflow
import mlflow.keras

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "file:///kaggle/working/mlruns")
if MLFLOW_TRACKING_URI.startswith("file://"):
    Path(MLFLOW_TRACKING_URI.replace("file://", "")).mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("ludus_cnn")
mlflow.keras.autolog(log_models=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 41.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 96.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.1/197.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 13.3 MB/s eta 0:00:00
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=6

2026-03-22 16:07:25.268706: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774195645.460706      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774195645.518910      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774195645.980944      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774195645.980988      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774195645.980991      17 computation_placer.cc:177] computation placer alr

In [2]:
# using a testing model to train faster and experiment with mlflow

from Backend.model_architecture.implementation.experimental.model_testing_mlflow import model_testing_mlflow
model = model_testing_mlflow(conv_filters=12, upsample_factor=4)

2026-03-22 16:07:50.532063: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [3]:
# deleting the folder to avoid chaos in the output
!rm -r ludus_scacchi

In [4]:
from keras.models import load_model

In [5]:
# how many cycles need to be done
# used to avoid timeout with heavier models 

MAX_N = 5
n = 5
starting_n = 0
conv_size = 32

model_name = 'lenet_baseline'
year_month = '2013-01'

mlflow_run_id = None

with mlflow.start_run(run_name=f"{model_name}_{year_month}") as run:
    mlflow_run_id = run.info.run_id
    mlflow.log_params({
        "model_name": model_name,
        "year_month": year_month,
        "max_n": MAX_N,
        "start_chunk": starting_n,
        "end_chunk": n - 1,
        "conv_size": conv_size,
        "kernel_size": (2, 2),
        "epochs_per_chunk": 3,
        "batch_size": 64
    })

    for i in range(starting_n, n):
        print("=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=")
        print("Starting training with chunk number: ", i)
        
        model.name = model_name + '_chunk_' + str(i) + '_' + year_month
        x = np.load('/kaggle/input/lichess-bitboards-2013-01/chunk_' + str(i) + '.npy')
        y = np.load('/kaggle/input/lichess-bitboards-2013-01/chunk_' + str(i) + '_y.npy')

        # leaving space for evaluation 
        if i == MAX_N - 1:
            x = x[:-10000]
            y = y[:-10000]

        x = np.transpose(x, (0, 2, 3, 1))
        
        y2 = []
        for j in y:
            y2.append(j[0].flat)
        y2 = np.array(y2)
        
        history = model.fit(x, y2, batch_size=64, epochs=3)
        mlflow.log_metric("chunk_index", i, step=i)
        model.save(model.name + '.keras')
        print("saved model with name: ", model.name)
    model.save(f'{model_name}.keras')


=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  0


Epoch 1/3
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 1112s 71ms/step - accuracy: 0.0714 - loss: 3.5871
Epoch 2/3
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 1139s 73ms/step - accuracy: 0.2577 - loss: 2.7208
Epoch 3/3
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 1164s 75ms/step - accuracy: 0.3455 - loss: 2.3932


2026/03/22 17:04:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


saved model with name:  lenet_baseline_chunk_0_2013-01
=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  1


Epoch 1/3
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 1177s 75ms/step - accuracy: 0.3737 - loss: 2.2739
Epoch 2/3
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 1307s 84ms/step - accuracy: 0.3885 - loss: 2.1993
Epoch 3/3
15625/15625 ━━━━━━━━━━━━━━━━━━━━ 1361s 87ms/step - accuracy: 0.3950 - loss: 2.1591


...

2026/03/22 20:04:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


saved model with name:  lenet_baseline_chunk_3_2013-01
=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=A=
Starting training with chunk number:  4


Epoch 1/3
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 1011s 65ms/step - accuracy: 0.4104 - loss: 2.0584
Epoch 2/3
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 1007s 65ms/step - accuracy: 0.4149 - loss: 2.0406
Epoch 3/3
15469/15469 ━━━━━━━━━━━━━━━━━━━━ 1010s 65ms/step - accuracy: 0.4154 - loss: 2.0330


2026/03/22 20:55:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


saved model with name:  lenet_baseline_chunk_4_2013-01


In [6]:
from pathlib import Path

if mlflow_run_id is None:
    raise RuntimeError("MLflow run was not started. Please re-run the training cell.")

run_id_path = Path('/kaggle/working/mlflow_run_id.txt')
run_id_path.write_text(mlflow_run_id)
print(f"Saved MLflow run id {mlflow_run_id} to {run_id_path}")
mlflow_run_id

Saved MLflow run id 269e1638d89c4f4890c4e0d8b80c10c4 to /kaggle/working/mlflow_run_id.txt


'269e1638d89c4f4890c4e0d8b80c10c4'